In [ ]:
# ---------------------------------------------------------
# 1. IMPORTAÇÃO DE BIBLIOTECAS
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
# ---------------------------------------------------------
# 2. CONFIGURAÇÃO (PREENCHA ESTA PARTE COM SEUS DADOS)
# ---------------------------------------------------------

# Lista com TODAS as variáveis métricas usadas nas 3 perspectivas
# (Ex: Gastos, Frequência, Recência, Score de Produto, etc.)
metric_features = [
    'Total_Spend_Month_1', 'Total_Spend_Month_2', 'Total_Spend_Month_3', # Exemplo Valor
    'Flights_per_day', 'RecencyInMonths', 'Days_in_prog',                # Exemplo Comportamento
    'Product_A_Spend', 'Product_B_Spend'                                 # Exemplo Produto (Adicione as suas)
]

# Lista com os nomes das colunas dos CLUSTERS que vocês já geraram
# Certifique-se que estas colunas existem no df_final
cluster_cols = [
    'rb_scaled_ward_cluster6',  # Seu cluster de Valor (do notebook anterior)
    'cluster_behavior',         # Nome da coluna do cluster de comportamento (do seu grupo)
    'cluster_product'           # Nome da coluna do cluster de produto (do seu grupo)
]

In [ ]:
# ---------------------------------------------------------
# 3. CRIAÇÃO DOS MICRO-SEGMENTOS (CENTRÓIDES)
# ---------------------------------------------------------

# Agrupar pelas perspectivas e calcular a média das métricas
# Isso cria uma linha para cada combinação única de clusters (Ex: Grupo 1 de Valor + Grupo 2 de Comp...)
df_centroids = df_final.groupby(cluster_cols)[metric_features].mean()

print(f"Número de Micro-segmentos (Combinações) gerados: {len(df_centroids)}")
display(df_centroids.head())

In [ ]:
# ---------------------------------------------------------
# 4. PADRONIZAÇÃO DOS CENTRÓIDES
# ---------------------------------------------------------
# É crucial padronizar os centróides antes de agrupar, pois as escalas das métricas são diferentes
scaler = StandardScaler()
centroids_scaled = scaler.fit_transform(df_centroids)

In [ ]:
# ---------------------------------------------------------
# 5. DENDROGRAMA (PARA DECIDIR O NÚMERO FINAL DE GRUPOS)
# ---------------------------------------------------------
plt.figure(figsize=(12, 6))
plt.title('Dendrograma dos Micro-Segmentos')
plt.xlabel('Combinações de Clusters')
plt.ylabel('Distância Euclidiana')

# Criar a matriz de ligação (linkage matrix) usando o método 'ward'
linkage_matrix = linkage(centroids_scaled, method='ward')

dendrogram(linkage_matrix, color_threshold=0) # color_threshold=0 mostra tudo da mesma cor por enquanto
plt.show()

In [ ]:
# ---------------------------------------------------------
# 6. APLICAÇÃO DO MERGE (CLUSTERING FINAL)
# ---------------------------------------------------------

# --- DECISÃO MANUAL ---
# Olhe para o Dendrograma acima e escolha um número de cortes (n_clusters)
# ou defina uma distância de corte.
# Vamos assumir 7 como exemplo, mas mude conforme sua análise visual.
K_FINAL = 7  

# Inicializar e rodar o modelo nos CENTRÓIDES
hc_final = AgglomerativeClustering(n_clusters=K_FINAL, linkage='ward')
final_labels = hc_final.fit_predict(centroids_scaled)

# Adicionar os labels finais ao dataframe de centróides
df_centroids['General_Cluster'] = final_labels

In [ ]:
# ---------------------------------------------------------
# 7. MAPEAMENTO DE VOLTA PARA OS CLIENTES (DF_FINAL)
# ---------------------------------------------------------

# Resetar o índice para transformar os índices (as combinações) em colunas normais
df_centroids_reset = df_centroids.reset_index()

# Manter apenas as colunas chave (cluster_cols) e o label final
df_merge_key = df_centroids_reset[cluster_cols + ['General_Cluster']]

# Fazer o merge com o dataframe original
# Isso vai atribuir o 'General_Cluster' correto para cada cliente com base na sua combinação
df_final = df_final.merge(df_merge_key, on=cluster_cols, how='left')

In [ ]:
# ---------------------------------------------------------
# 8. ANÁLISE DOS RESULTADOS
# ---------------------------------------------------------

print("Distribuição dos Clientes por Cluster Geral:")
print(df_final['General_Cluster'].value_counts().sort_index())

# Visualização Rápida (Heatmap dos Centróides Finais)
# Recalcular a média agora baseada no NOVO cluster geral
final_centroids = df_final.groupby('General_Cluster')[metric_features].mean()
final_centroids_scaled = scaler.fit_transform(final_centroids) # Apenas para visualização no heatmap

plt.figure(figsize=(15, 8))
sns.heatmap(final_centroids_scaled.T, cmap='RdBu', center=0, annot=True, fmt='.1f')
plt.title('Perfil dos Clusters Finais (Valores Padronizados)')
plt.xlabel('Cluster Geral')
plt.ylabel('Variáveis Métricas')
plt.show()